# Discrete excitable media

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/zoo/excitable_media.ipynb)

*Syga S, Nava-Sedeño JM, Brusch L, Deutsch A (2019). A lattice-gas
cellular automaton model for discrete excitable media. In: Tsuji K, Müller
SC (eds) Spirals and Vortices. The Frontiers Collection, pp 253–264.
Springer. [doi:10.1007/978-3-030-05798-5_15](https://doi.org/10.1007/978-3-030-05798-5_15)*

**Question.** Spiral waves organise excitable media from the aggregation
of *Dictyostelium* to heart tissue. Continuum models describe them well,
but many biological excitable media consist of a small number of discrete
individuals. Do spirals survive the fluctuations of a discrete medium, and
how do they differ from those of the continuum model?

The notebook reproduces Fig 2 of the chapter: a spiral wave in the LGCA and
in its mean field, Barkley's model, from the same start; the nullclines of
the reactions; and the distribution of the LGCA's node states around the
orbit of the mean field.

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import lgca
from lgca import zoo
from lgca.model import run_model
from lgca.zoo import excitable_media as excitable

zoo.parameter_table("excitable_media")

## The model

Two species share a hexagonal lattice. The excited species $X$ lives in
the $b = 6$ velocity channels and moves; the refractory species $Y$ lives in
the $a = K - 6$ rest channels and stays. With $\rho_X$ and $\rho_Y$ the
occupied fractions of the velocity and of the rest channels of a node, in
every time step

1. $Y$ reacts once: one $Y$ cell is added with probability $\rho_X$ and one
   removed with probability $\rho_Y$;
2. $X$ reacts $N$ times: one $X$ cell is added with probability
   $\rho_X^2\,(1 + (\rho_Y + B)/A)$ and one removed with probability
   $\rho_X(\rho_Y + B)/A + \rho_X^3$;
3. the $X$ cells are mixed over the velocity channels, then move.

The mean changes per reaction are Barkley's reaction terms,

$$f = \rho_X(1 - \rho_X)\Big(\rho_X - \frac{\rho_Y + B}{A}\Big), \qquad g = \rho_X - \rho_Y,$$

so that $X$ is fast and excitable and $Y$ slow and refractory. The number
of rest channels sets the size of the fluctuations: $\rho_Y$ changes in
steps of $1/a$. The rule is the built-in `excitable_medium`:

In [ ]:
spec = excitable.build_spec()  # A = 0.75, B = 0.02, N = 50, K = 23 as in Fig 2
print(spec.dynamics.operators[0], spec.space)
run = run_model(spec, showprogress=False)
nodes_t, steps = np.asarray(run.data["nodes"]), list(run.data.steps("nodes"))

## The mean field

In the LGCA's own units, one time step and one node, the mean field is

$$\partial_k\rho_X = D\,\Delta\rho_X + \frac{N}{b}\,f, \qquad \partial_k\rho_Y = \frac{g}{a},$$

with $D = 1/4$: every $X$ cell moves one node per step in one of six
directions, and each of the $N$ reactions changes the number of $X$ cells,
$b\rho_X$, by $f$ on average. Rescaling time and space gives Barkley's model
in the form of the chapter (its Eq 8). `excitable.barkley` integrates it on
the same lattice, from the same start, with $\rho_X = 0$ beyond the
absorbing edge:

In [ ]:
x_0, y_0 = excitable.fractions(nodes_t[0])
centre = tuple(size // 2 for size in run.lgca.dims)
pde = excitable.barkley(run.lgca, x_0, y_0, steps=spec.time.steps, probe=centre)


def draw(axis, rho_x, rho_y, title):
    """Red for X, green for Y, yellow where both are high, on the hexagonal lattice."""
    lattice = run.lgca
    axis.scatter(lattice.xcoords.ravel(), lattice.ycoords.ravel(), c=excitable.colour(rho_x, rho_y).reshape(-1, 3),
                 marker="h", s=9, lw=0)
    axis.set(aspect="equal", xticks=[], yticks=[], title=title)
    axis.set_facecolor("black")


times = (50, 150, 300, 600)
fig, axes = plt.subplots(2, 4, figsize=(16, 8.4), constrained_layout=True)
for column, step in enumerate(times):
    draw(axes[0, column], *excitable.fractions(nodes_t[steps.index(step)]), f"(a) LGCA, $k = {step}$")
    index = list(pde["steps"]).index(step)
    draw(axes[1, column], pde["x"][index], pde["y"][index], f"(b) mean field, $k = {step}$")
plt.show()
plt.close(fig)

At first the two models agree: the broken front of the initial
condition curls into a single spiral with the same wavelength and period.
At long times the fluctuations of the LGCA break the spiral into several
smaller ones, while the mean field keeps rotating. The chapter observes the
same: spiral break-up, common in stochastic models of excitable media.

## Nullclines and the distribution of node states (Fig 2c, d)

In [ ]:
A, B = 0.75, 0.02
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
rho = np.linspace(0, 1, 200)
axes[0].axvline(0, color="tab:blue")
axes[0].axvline(1, color="tab:blue")
axes[0].plot(rho, A * rho - B, color="tab:blue", label="$f = 0$")
axes[0].plot(rho, rho, color="tab:red", label="$g = 0$")
grid_x, grid_y = np.meshgrid(np.linspace(0.05, 0.95, 12), np.linspace(0.05, 0.95, 12))
f, _ = excitable.reaction(grid_x, grid_y, A, B)
axes[0].quiver(grid_x, grid_y, np.sign(f), 0, color="0.5", scale=30)  # the fast direction of X
axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="$\\rho_X$", ylabel="$\\rho_Y$", title="(c) nullclines")
axes[0].legend(loc="upper left")

late = np.asarray([step >= 200 for step in steps])  # after a transient, as in the chapter
x, y = excitable.fractions(nodes_t[late])
histogram, xedges, yedges = np.histogram2d(x.ravel(), y.ravel(), bins=[7, 18], range=[[0, 1], [0, 1]])
axes[1].pcolormesh(xedges, yedges, histogram.T / histogram.sum(), cmap="Greys", vmax=0.025)
orbit = pde["probe"][300:]  # one node of the mean field, every step, after the spiral has formed
axes[1].plot(orbit[:, 0], orbit[:, 1], color="tab:orange", label="mean field, one node")
axes[1].set(xlim=(0, 1), ylim=(0, 1), xlabel="$\\rho_X$", ylabel="$\\rho_Y$",
            title="(d) LGCA node states (grey) and the mean field's orbit")
axes[1].legend(loc="upper left")
plt.show()
plt.close(fig)

The only stable state is $\rho_X = \rho_Y = 0$; the middle branch of
$f = 0$ is the excitation threshold. A perturbation beyond it makes $X$
saturate (the arrows), $Y$ follows slowly and pulls $X$ back down: the
excitation loop. The states of the LGCA's nodes (grey, the frequency of
each pair $(\rho_X, \rho_Y)$, whose values are multiples of $1/6$ and
$1/17$) follow the same loop as the mean field, spirals and broken spirals
alike.

## Differences from the chapter

- **Lattice size and run length** are not given in the chapter: 100 × 100
  nodes and 600 steps here, 200 × 200 and 2000 with `full=True`.
- **The mean field** is written in the LGCA's units, with the rates
  $N/b$ and $1/a$; the chapter's Eq 8 is the same model after rescaling. We
  integrate it with explicit Euler steps on the lattice itself, instead of
  the program of Barkley and Dowle.
- **The histogram** covers the frames of one run after 200 steps, not
  $8 \cdot 10^5$ nodes of many realizations after 2000 steps.

## Explore it

Move the excitability $A$ and the threshold $B$ while the spirals turn:
below a threshold of $A$ the excitation dies out (the density view shows
all cells, excited and refractory).

In [ ]:
explorer = lgca.explore(
    excitable.build_spec(size=80),
    {"alpha": (0.3, 1.0), "beta": (0.0, 0.1)},  # A and B
    view="density",
    steps_per_frame=2,
)
explorer

## Things to try

1. **Fluctuations.** Rerun with $K = 12$ and $K = 60$
   (`excitable.build_spec(K=12)`). When do the spirals break up?
2. **Spirals where the continuum has none (Fig 3).** With $B = 0.05$,
   $N = 50$ and $K = 12$, vary $A$ from 0.2 to 1 and measure the mean return
   time of a node (`excitable.mean_return_time` on states recorded every
   step): the chapter finds spirals at smaller $A$ in the LGCA than in
   Barkley's model, an effect of the fluctuations.
3. **A single pulse.** Start from a line of excited nodes instead of the
   quadrants: does the pulse travel without spiralling?